[![Open Rendered Output](https://img.shields.io/badge/Rendered%20Output-Open-blue?logo=link&logoColor=white)](https://htmlpreview.github.io/?https://raw.githubusercontent.com/MeteoSwiss/nwp-fdb-polytope-demo/main/examples/snapshots/full_field.html)

# ICON-CH1-EPS Polytope retrieval and interpolation to the Swiss LV95 grid

> **Preview / not yet part of the default `earthkit-geo` dependency stack**
>
> `earthkit-geo==1.1.2` is released, but it still pins the older `eckit` and `mir-python` backend versions. For this Swiss-grid interpolation test we therefore install the newer backend wheels explicitly.

This notebook retrieves one full ICON-CH1-EPS field from MeteoSwiss Polytope and interpolates it to the predefined 1-km Swiss LV95 grid (`EPSG:2056`).

## Environment setup

Create a clean Python 3.11 environment and install the released Earthkit packages first:

```bash
python3.11 -m venv .venv-ch1-swissgrid
source .venv-ch1-swissgrid/bin/activate
python -m pip install --upgrade pip setuptools wheel

python -m pip install \
    "earthkit-data[polytope]==1.1.0" \
    earthkit-geo==1.1.2 \
    pyproj==3.7.2 \
    eccodes-cosmo-resources-python==2.47.0.1 \
    ipykernel
```

Then override the regridding backend with the newer PyPI wheels:

```bash
python -m pip install \
    eckit==2.3.0.30 \
    mir-python==1.30.1.30
```

The matching native libraries are pulled in transitively by these packages.

Register the environment as a Jupyter kernel if needed:

```bash
python -m ipykernel install --user \
    --name ch1-swissgrid \
    --display-name "Python (CH1 → Swiss LV95)"
```

In [ ]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "earthkit-data",
    "earthkit-geo",
    "eckit",
    "eckitlib",
    "mir-python",
    "mirlib",
    "eccodeslib",
    "atlaslib-ecmwf",
    "eccodes-cosmo-resources-python"
]

for package in packages:
    try:
        print(f"{package:<20}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:<20}: NOT INSTALLED")

## Configuring Access to Polytope
To access ICON data via MeteoSwiss's Polytope, you need a Polytope offline token provided by MeteoSwiss. If you do not already have a token, you can request one [here](https://meteoswiss.atlassian.net/wiki/spaces/IW2/pages/327780397/Polytope#Offline-token-authentication). Then, create a new `config.yml` file based on [`config_example.yml`](config_example.yml), and replace <meteoswiss_key> with your access token there. 

In [ ]:
import yaml
import os

def load_config(path="config.yml"):
    if not os.path.exists(path):
        raise FileNotFoundError("Missing config.yml. Please create one based on config_example.yml")
    with open(path, "r") as f:
        return yaml.safe_load(f)

config = load_config()

#ICON-CSCS Polytope credentials
os.environ["POLYTOPE_USER_KEY"] = config["meteoswiss"]["key"]
os.environ["POLYTOPE_ADDRESS"] = "https://service.meteoswiss.ch/polytope"

⚙️ eccodes Definitions Setup
Moreover, MeteoSwiss uses custom ecCodes definitions for ICON data. To correctly interpret MeteoSwiss-specific shortName values, you must configure the definition path.

In [ ]:
import os
os.environ['ECCODES_ECKIT_GEO'] = "1"  # before importing eccodes, enable ICON native grid support
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"  # disable warning about eccodes version mismatch
import eccodes
import eccodes_cosmo_resources

vendor = eccodes.codes_definition_path()
cosmo = eccodes_cosmo_resources.get_definitions_path()
eccodes.codes_set_definitions_path(f"{cosmo}:{vendor}")

## Retrieve one full ICON-CH1-EPS field from Polytope

This uses the `mch` collection because no geographic feature/subset is requested.

In [ ]:
import earthkit.data as ekd

request = {
    "param": "500011", # T_2M
    "date": "20260929",
    "time": "0600",
    "expver": "0001",
    "step": 0,
    "class": "od",
    "levtype": "sfc",
    "model": "icon-ch1-eps",
    "stream": "enfo",
    "type": "CF",
    "timespan": "none",
}

ds = ekd.from_source(
    "polytope",
    "mch",
    request,
    stream=False,
)

fl = ds.to_fieldlist()
field = fl[0]
field

## Inspect the native ICON-CH1-EPS grid

In [ ]:
print(field.ls())
print("gridType:", field.metadata("gridType"))
print("grid spec:", field.geography.grid_spec())

values = field.to_numpy()
print("number of source values:", values.size)

## Build the Swiss LV95 target grid

The predefined `swisslv95` grid is a regular 1-km grid in `EPSG:2056`.

In [ ]:
from eckit.geo import Grid

outgrid = Grid({"grid": "swisslv95"})

print("Swiss LV95 grid specification:")
print(outgrid.spec)
print("shape:", outgrid.shape)
print("size:", outgrid.size())

## Convert the Swiss target to latitude/longitude points

The Swiss LV95 grid is a regular projected grid. For interpolation, we obtain its latitude/longitude coordinates and pass those points to earthkit-geo as an unstructured_ll target.

In [ ]:
out_lats, out_lons = outgrid.to_latlons()

unstructured_outgridspec = {
    "type": "unstructured_ll",
    "latitudes": out_lats,
    "longitudes": out_lons,
}

print("number of target points:", len(out_lats))

## Interpolate ICON-CH1 to the Swiss LV95 grid

In [ ]:
import earthkit.geo as ekg

result, result_grid = ekg.grids.array.regrid(
    values,
    in_grid=field.geography.grid_spec(),
    out_grid=unstructured_outgridspec,
    interpolation="linear",
)

print("result shape:", result.shape)
print("result grid:", result_grid)

## Sanity checks

In [ ]:
import numpy as np

print(f"Source points: {values.size}")
print(f"Target points: {result.size}")

print(
    f"Source range: {np.nanmin(values):.2f} – "
    f"{np.nanmax(values):.2f} K"
)
print(
    f"Result range: {np.nanmin(result):.2f} – "
    f"{np.nanmax(result):.2f} K"
)

print(f"Source NaNs: {np.isnan(values).sum()}")
print(f"Result NaNs: {np.isnan(result).sum()}")